# Chat Model Event Streams

The `chat_model_stream.py` module defines per-message streaming objects for version 3 chat-model event streams.

`ChatModelStream` is the synchronous stream returned by `BaseChatModel.stream_events(version="v3")`. `AsyncChatModelStream` is the asynchronous stream returned by `BaseChatModel.astream_events(version="v3")`.

Both stream variants:

- Consume content-block protocol events.
- Accumulate text, reasoning, tool-call, usage, metadata, and content-block state.
- Expose typed projections for text, reasoning, tool calls, and the final output.
- Preserve raw protocol events in a replay buffer.
- Support multiple independent consumers beginning from the first buffered event.
- Assemble a final `AIMessage` whose content blocks use the version 1 protocol shape.

## Public Exports

The module explicitly exports the following classes:

```python
__all__ = [
    "AsyncChatModelStream",
    "AsyncProjection",
    "ChatModelStream",
    "SyncProjection",
    "SyncTextProjection",
]
```

# Shared Projection Interface

`SyncProjection`, `SyncTextProjection`, and `AsyncProjection` inherit common producer-side lifecycle behaviour from the internal `_ProjectionBase`.

The producer pushes deltas into the projection and eventually either completes it with a final value or fails it with an exception.

### Properties

1. `done`: Indicates whether the projection has completed successfully or failed.
   * **Type:**
     ```python
     done: bool
     ```

2. `error`: Returns the terminal exception when the projection failed.
   * **Type:**
     ```python
     error: BaseException | None
     ```

### Methods

1. `push`: Appends one delta to the projection's replay buffer.

   Existing consumers and later consumers can observe the delta. This is primarily a producer-side method used by the owning stream.

   * **Syntax:**
     ```python
     push(
         self,
         delta: Any # Delta value to append
     ) -> None
     ```

2. `complete`: Stores the final accumulated value and marks the projection as successfully finished.
   * **Syntax:**
     ```python
     complete(
         self,
         final_value: Any # Final accumulated projection value
     ) -> None
     ```

3. `fail`: Stores a terminal exception and marks the projection as finished.

   Consumers raise the stored exception when they next attempt to drain or iterate the projection.

   * **Syntax:**
     ```python
     fail(
         self,
         error: BaseException # Terminal projection error
     ) -> None
     ```

# SyncProjection

`SyncProjection` is a synchronous replayable iterable of delta values.

When a consumer reaches the end of the current buffer and the projection has not finished, the projection can invoke a configured pull callback to request another source event. Each new iterator starts at buffer position zero.

## Bases

- `_ProjectionBase`

### Methods

1. `__init__`: Creates an empty synchronous projection with no lazy-start or pull callback.
   * **Syntax:**
     ```python
     __init__(
         self
     ) -> None
     ```

2. `set_start`: Installs a callback invoked before the projection is consumed for the first time.

   Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_start(
         self,
         cb: Callable[
             [],
             None
         ] | None # Lazy-start callback
     ) -> None
     ```

3. `set_request_more`: Installs the pull callback used when a consumer catches up with the current delta buffer.

   The callback returns `True` when another source event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_request_more(
         self,
         cb: Callable[
             [],
             bool
         ] | None # Source-pump callback
     ) -> None
     ```

4. `__iter__`: Returns a synchronous iterator over all buffered and subsequently produced deltas.

   Each call starts from the first buffered delta. When no pull callback is installed, iteration stops after the currently available values unless the projection is already complete.

   The stored terminal exception is raised when the projection failed.

   * **Syntax:**
     ```python
     __iter__(
         self
     ) -> Iterator[Any]
     ```

5. `get`: Drains the source through the pull callback and returns the final accumulated value.

   The lazy-start callback is invoked first when configured. The stored terminal exception is raised when the projection failed.

   If the source becomes exhausted without calling `complete`, the currently stored final value is returned, which may be `None`.

   * **Syntax:**
     ```python
     get(
         self
     ) -> Any
     ```

# SyncTextProjection

`SyncTextProjection` specializes `SyncProjection` for text and reasoning content.

It retains normal delta iteration while adding string conversion, truth-value testing, and a text-oriented representation.

## Bases

- `SyncProjection`

### Methods

1. `__str__`: Drains the projection and returns the complete accumulated string.

   An unset or `None` final value is represented as an empty string.

   * **Syntax:**
     ```python
     __str__(
         self
     ) -> str
     ```

2. `__bool__`: Returns whether at least one delta has been pushed.

   This check does not drain the projection.

   * **Syntax:**
     ```python
     __bool__(
         self
     ) -> bool
     ```

3. `__repr__`: Returns a representation of the final value when complete.

   Before completion, it returns a representation of the currently concatenated delta strings without requesting further events.

   * **Syntax:**
     ```python
     __repr__(
         self
     ) -> str
     ```

# AsyncProjection

`AsyncProjection` is an asynchronous replayable iterable that is also awaitable for its final value.

An `asyncio.Event` notifies consumers when projection state changes. Producers and consumers must use the same event loop.

Each asynchronous iterator has an independent cursor beginning at the first buffered delta.

## Bases

- `_ProjectionBase`

### Methods

1. `__init__`: Creates an empty asynchronous projection with no lazy-start or pull callback.
   * **Syntax:**
     ```python
     __init__(
         self
     ) -> None
     ```

2. `set_start`: Installs an asynchronous callback invoked before first consumption.

   Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_start(
         self,
         cb: Callable[
             [],
             Awaitable[None]
         ] | None # Asynchronous lazy-start callback
     ) -> None
     ```

3. `set_arequest_more`: Installs the asynchronous pull callback used when a consumer catches up with the current buffer.

   The callback returns `True` when a new source event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_arequest_more(
         self,
         cb: Callable[
             [],
             Awaitable[bool]
         ] | None # Asynchronous source-pump callback
     ) -> None
     ```

4. `push`: Appends one delta and wakes waiting consumers.
   * **Syntax:**
     ```python
     push(
         self,
         delta: Any # Delta value to append
     ) -> None
     ```

5. `complete`: Stores the final accumulated value, marks the projection complete, and wakes waiting consumers.
   * **Syntax:**
     ```python
     complete(
         self,
         final_value: Any # Final accumulated projection value
     ) -> None
     ```

6. `fail`: Stores a terminal exception, marks the projection finished, and wakes waiting consumers.
   * **Syntax:**
     ```python
     fail(
         self,
         error: BaseException # Terminal projection error
     ) -> None
     ```

7. `__aiter__`: Returns a new asynchronous iterator over projection deltas.

   Each iterator replays the complete delta buffer from the beginning and then waits for or requests additional values.

   * **Syntax:**
     ```python
     __aiter__(
         self
     ) -> AsyncIterator[Any]
     ```

8. `__await__`: Makes the projection awaitable for its final accumulated value.

   When a pull callback is configured, awaiting the projection drives that callback until completion or source exhaustion. Otherwise, it waits for producer notifications.

   The stored terminal exception is raised when the projection failed.

   * **Syntax:**
     ```python
     __await__(
         self
     ) -> Generator[
         Any,
         None,
         Any
     ]
     ```

# Shared Chat Model Stream Interface

`ChatModelStream` and `AsyncChatModelStream` inherit shared event accumulation and metadata behaviour from the internal `_ChatModelStreamBase`.

The shared implementation processes message and content-block protocol events, accumulates typed values, finalizes tool-call chunks, and assembles the final `AIMessage`.

### Properties

1. `namespace`: Returns the graph namespace path associated with the streamed message.
   * **Type:**
     ```python
     namespace: list[str]
     ```

2. `node`: Returns the graph node that produced the message.
   * **Type:**
     ```python
     node: str | None
     ```

3. `message_id`: Returns the stable message identifier when one is available.
   * **Type:**
     ```python
     message_id: str | None
     ```

4. `done`: Indicates whether the stream has completed or failed.
   * **Type:**
     ```python
     done: bool
     ```

5. `has_events`: Indicates whether at least one raw protocol event has been recorded.
   * **Type:**
     ```python
     has_events: bool
     ```

6. `output_message`: Returns the assembled message after successful stream completion.

   This property never starts the source, pumps events, blocks, or raises the stored stream error. It returns `None` while no output message is available.

   * **Type:**
     ```python
     output_message: AIMessage | None
     ```

### Methods

1. `set_message_id`: Assigns the stable message identifier after the underlying chat-model run starts.

   This method is intended for the stream driver rather than normal consumer code.

   * **Syntax:**
     ```python
     set_message_id(
         self,
         message_id: str # Stable message identifier
     ) -> None
     ```

2. `dispatch`: Processes one content-block protocol event.

   Supported event types include:

   - `"message-start"`
   - `"content-block-delta"`
   - `"content-block-finish"`
   - `"message-finish"`
   - `"error"`

   Every event is first appended to the raw replay buffer. A `"content-block-start"` event is retained in the buffer but requires no accumulation work.

   An `"error"` event fails the stream with a `RuntimeError`.

   * **Syntax:**
     ```python
     dispatch(
         self,
         event: Mapping[
             str,
             Any
         ] # Content-block protocol event
     ) -> None
     ```

3. `fail`: Marks the stream as failed and propagates the exception to its projections.

   The asynchronous stream additionally fails its output and raw-event projections.

   * **Syntax:**
     ```python
     fail(
         self,
         error: BaseException # Terminal stream error
     ) -> None
     ```

## Event Accumulation

The shared implementation performs the following processing:

- Text deltas are accumulated globally and by content-block index.
- Reasoning deltas are accumulated globally and by content-block index.
- Tool-call chunks retain their first non-empty ID and name while argument fragments are accumulated.
- Server-side tool-call chunks are accumulated separately and do not appear in the public `tool_calls` projection.
- Tool-call argument strings are parsed when their blocks or message finish.
- Unparseable tool calls become invalid tool calls.
- Finished blocks are retained by event index so final content ordering matches protocol ordering.
- Message-start metadata may provide the model provider, model name, and message ID.
- Message-finish data may provide usage metadata, response metadata, and provider-specific additional keyword arguments.

## Final Output Assembly

The final `AIMessage` contains:

- Ordered finalized content blocks.
- The stable message ID.
- Finalized client-side tool calls.
- Invalid tool calls.
- Usage metadata.
- Response metadata.
- Optional provider-specific additional keyword arguments.

When protocol blocks were received, `AIMessage.content` is a list of version 1 content-block dictionaries. When no protocol blocks were received, accumulated text is used as string content.

The output metadata always sets:

```python
response_metadata["output_version"] = "v1"
```

This prevents already normalized version 1 blocks from being translated again by another output-version handler.

# ChatModelStream

`ChatModelStream` is the synchronous per-message stream returned by `BaseChatModel.stream_events(version="v3")`.

It exposes synchronous projections, direct raw-event iteration, and a blocking `output` property.

## Bases

- `_ChatModelStreamBase`

### Methods

1. `__init__`: Creates an empty synchronous chat-model stream.

   Text and reasoning projections are created as `SyncTextProjection` objects. Tool calls use a `SyncProjection`.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         namespace: list[str] | None = None, # Graph namespace path
         node: str | None = None, # Graph node producing the message
         message_id: str | None = None # Initial stable message identifier
     ) -> None
     ```

2. `bind_pump`: Binds a standalone source-pump callback.

   This delegates to `set_request_more` and is used by synchronous version 3 chat-model event streaming.

   * **Syntax:**
     ```python
     bind_pump(
         self,
         pump_one: Callable[
             [],
             bool
         ] # Callback that requests one additional source event
     ) -> None
     ```

3. `set_start`: Installs a lazy-start callback on the stream and all synchronous projections.

   Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_start(
         self,
         cb: Callable[
             [],
             None
         ] | None # Lazy-start callback
     ) -> None
     ```

4. `set_request_more`: Installs one source-pump callback on the stream and all synchronous projections.

   The callback should return `True` when another event was produced and `False` when the source is exhausted.

   * **Syntax:**
     ```python
     set_request_more(
         self,
         cb: Callable[
             [],
             bool
         ] # Shared source-pump callback
     ) -> None
     ```

5. `__iter__`: Returns a synchronous iterator over raw protocol events.

   Every iterator starts at the beginning of the replay buffer. When it catches up, it requests more events through the configured pump until the stream completes or the source is exhausted.

   The stored terminal exception is raised when the stream failed.

   * **Syntax:**
     ```python
     __iter__(
         self
     ) -> Iterator[MessagesData]
     ```

### Properties

1. `text`: Returns the cached text projection.

   Iterating it yields text deltas. Converting it with `str()` drains the stream and returns the complete text.

   * **Type:**
     ```python
     text: SyncTextProjection
     ```

2. `reasoning`: Returns the cached reasoning projection.

   It has the same behaviour as `text`.

   * **Type:**
     ```python
     reasoning: SyncTextProjection
     ```

3. `tool_calls`: Returns the cached tool-call projection.

   Iteration yields `ToolCallChunk` deltas. Calling `get()` returns the finalized `list[ToolCall]`.

   * **Type:**
     ```python
     tool_calls: SyncProjection
     ```

4. `output`: Drains the remaining source events and returns the assembled message.

   The stored stream exception is raised when the stream failed. A `RuntimeError` is raised when the source finishes without producing a message.

   * **Type:**
     ```python
     output: AIMessage
     ```

# AsyncChatModelStream

`AsyncChatModelStream` is the asynchronous per-message stream returned by `BaseChatModel.astream_events(version="v3")`.

It is both awaitable for the final `AIMessage` and asynchronously iterable over raw protocol events. Its projections are individually asynchronous iterables and awaitables.

## Bases

- `_ChatModelStreamBase`

### Methods

1. `__init__`: Creates an empty asynchronous chat-model stream.

   Text, reasoning, tool-call, output, and raw-event projections are created as `AsyncProjection` objects.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         namespace: list[str] | None = None, # Graph namespace path
         node: str | None = None, # Graph node producing the message
         message_id: str | None = None # Initial stable message identifier
     ) -> None
     ```

2. `set_arequest_more`: Installs one asynchronous source-pump callback on every projection.

   The callback should return `True` when another source event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_arequest_more(
         self,
         cb: Callable[
             [],
             Awaitable[bool]
         ] | None # Shared asynchronous source-pump callback
     ) -> None
     ```

3. `set_start`: Installs one asynchronous lazy-start callback on the stream and every projection.

   Passing `None` removes the callback.

   * **Syntax:**
     ```python
     set_start(
         self,
         cb: Callable[
             [],
             Awaitable[None]
         ] | None # Asynchronous lazy-start callback
     ) -> None
     ```

4. `__await__`: Makes the stream awaitable for the assembled `AIMessage`.

   After the output projection resolves, the method also awaits the producer task so post-stream work, including completion callbacks, has finished before returning.

   * **Syntax:**
     ```python
     __await__(
         self
     ) -> Generator[
         Any,
         None,
         AIMessage
     ]
     ```

5. `__aiter__`: Returns an asynchronous iterator over raw protocol events.

   The iterator uses the raw-event replay projection, so each consumer begins at the first retained event.

   * **Syntax:**
     ```python
     __aiter__(
         self
     ) -> AsyncIterator[MessagesData]
     ```

6. `aclose`: Cancels or awaits the background producer and releases stream resources.

   When the stream has not yet produced a successful output, an active producer task is cancelled. The stream then fails with `asyncio.CancelledError`.

   When the output was produced successfully but post-stream work is still running, the task is awaited rather than cancelled so tracing completion callbacks are preserved.

   The method is idempotent and can be called before, during, or after normal completion.

   * **Syntax:**
     ```python
     async aclose(
         self
     ) -> None
     ```

7. `__aenter__`: Enters the asynchronous context manager and returns the stream.
   * **Syntax:**
     ```python
     async __aenter__(
         self
     ) -> Self
     ```

8. `__aexit__`: Exits the asynchronous context manager by calling `aclose`.
   * **Syntax:**
     ```python
     async __aexit__(
         self,
         exc_type: type[
             BaseException
         ] | None, # Exception type from the context
         exc: BaseException | None, # Exception raised inside the context
         tb: object # Traceback object
     ) -> None
     ```

### Properties

1. `text`: Returns the text projection.

   Asynchronous iteration yields text deltas. Awaiting it returns the complete text.

   * **Type:**
     ```python
     text: AsyncProjection
     ```

2. `reasoning`: Returns the reasoning projection.

   It has the same asynchronous iterable and awaitable behaviour as `text`.

   * **Type:**
     ```python
     reasoning: AsyncProjection
     ```

3. `tool_calls`: Returns the tool-call projection.

   Asynchronous iteration yields `ToolCallChunk` deltas. Awaiting it returns the finalized tool-call list.

   * **Type:**
     ```python
     tool_calls: AsyncProjection
     ```

4. `output`: Returns the output projection.

   Awaiting it returns the assembled `AIMessage`. Awaiting the complete stream instead also waits for the producer task and post-stream callbacks.

   * **Type:**
     ```python
     output: AsyncProjection
     ```

## Internal Components Omitted

The following underscore-prefixed implementation details are omitted as public API entries:

- Tool-call and content-block merge helpers
- `_ProjectionBase`
- `_AsyncProjectionIterator`
- `_ChatModelStreamBase`
- Per-block text and reasoning reconciliation helpers
- Internal event-recording and message-finalization hooks
- Internal output-message assembly helper